# 16. Build the final report

Regenerate the English LaTeX report strictly from persisted pipeline outputs, and show which files the renderer consumes so that every reported number is traceable to a CSV or JSON artefact.

**Reads**

- `outputs/tables/*.csv`
- `outputs/metrics/analysis_summary.json`
- `data/processed/*.csv`

**Writes**

- `report/report.tex`

**Method reference:** `METHODOLOGY.md` section 16

In [ ]:
"""Notebook environment: locate the repository and expose its data layers."""

import sys
import warnings
from pathlib import Path

import pandas as pd
from IPython.display import display

# Resolve the repository root from wherever the kernel was started, so the
# notebook works both from the repository root and from the notebooks directory.
ROOT = Path.cwd()
while not (ROOT / 'pyproject.toml').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if str(ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(ROOT / 'src'))

RAW = ROOT / 'data' / 'raw'
INTERIM = ROOT / 'data' / 'interim'
PROCESSED = ROOT / 'data' / 'processed'
TABLES = ROOT / 'outputs' / 'tables'
METRICS = ROOT / 'outputs' / 'metrics'

warnings.filterwarnings('ignore', category=UserWarning, module='openpyxl')
pd.set_option('display.max_columns', 80)
pd.set_option('display.width', 200)

print('repository:', ROOT.name)
print('pipeline outputs present:', (PROCESSED / 'fiscal_balances_1977_2025.csv').exists())

## 1. Render

The renderer reads persisted outputs and writes LaTeX. It contains no analysis:
if a number is in the report, it is in a CSV or JSON file first. That is what
makes the report checkable without running any code.

In [ ]:
from portugal_fiscal_balance.reporting.render import render_report

report_path = render_report(ROOT)
text = report_path.read_text(encoding='utf-8')
print('written:', report_path.relative_to(ROOT).as_posix())
print('size:', f'{len(text) / 1024:.1f} kB')
print('lines:', len(text.splitlines()))

## 2. Inputs the report consumes

In [ ]:
consumed = pd.DataFrame(
    [
        {
            'artefact': str(path.relative_to(ROOT)).replace('\\', '/'),
            'size_kb': round(path.stat().st_size / 1024, 1),
        }
        for path in [
            *sorted((ROOT / 'outputs' / 'tables').glob('*.csv')),
            *sorted((ROOT / 'outputs' / 'metrics').glob('*.json')),
        ]
    ]
)
display(consumed)

## 3. Structure of the generated report

In [ ]:
sections = [line.strip() for line in text.splitlines() if line.startswith('\\section')]
print(len(sections), 'sections')
for line in sections:
    print(' -', line.removeprefix('\\section{').removesuffix('}'))

In [ ]:
figure_lines = [line.strip() for line in text.splitlines() if 'includegraphics' in line]
print(len(figure_lines), 'figures included from outputs/figures')
for line in figure_lines:
    print(' -', line.split('{')[-1].removesuffix('}'))

## 4. First page of the source

In [ ]:
print(text[:2500])

## Interpretation limits

1. The report **restates persisted results**. It introduces no new calculation
   and no conclusion that is not supported by an artefact in `outputs/`.
2. It carries the **same caveats** as the notebooks: the 1995 splice, the
   1996-1999 component gap, and the descriptive nature of every regression.
3. Regenerating the report without first running the pipeline reproduces the
   **previous** outputs, because the renderer reads files rather than recomputing
   them.

---

[Previous: 15. European benchmark](15_european_benchmark.ipynb)

Every table shown above is also persisted as CSV, so results can be checked without reading notebook state. To rebuild everything from the bundled raw sources:

```bash
poetry install
make all
```